Este notebook toma sellout_analytical_dataset.csv como entrada y produce los hallazgos y visualizaciones para el avance de mi trabajo final, ya no se van a limpiar ni preparar datos.

#Exploratory Analysis: Sell Out de Refacciones

**Objetivo**. Analizar el dataset integrado de ventas e inventario para identificar patrones de comportamiento comercial y oportunidades relacionadas con productos, inventario y clientes.

El análisis se conecntrará en tres preguntas:
1. ¿Qué productos presetan mayor movimiento?
2. ¿Qué productos tienen inventario disponible pero poco o ningún movimiento?
3. ¿Qué clinetes concentran el mayor valor de compra?

Los resultados de este notebook corresponden a un análisis exploratorio del prototipo y no represetnan reglas comeciales definitivas

In [0]:
#Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [0]:
#Cargar dataset procesado
PROCESSED_PATH = Path("../data/processed")

analytical_file = (
    PROCESSED_PATH
    / "sellout_analytical_dataset.csv"
)

df = pd.read_csv(
    analytical_file,
    dtype={
        "cd_NumParte": "string",
        "cd_IdClienteFC": "string"
    }
)

print(
    f"Dataset analítico: "
    f"{df.shape[0]} filas, "
    f"{df.shape[1]} columnas"
)

display(df.head(10))

#Product Analysis

**¿Qué productos presentan mayor movimiento?**
Movimiento = Cantidad total vendida por número de parte


In [0]:
#Crear importe de venta
df["im_ImporteVenta"] = (
    df["ct_Cantidad"]
    * df["im_PrecioVenta"]
)

display(
    df[
        [
            "cd_NumParte",
            "ct_Cantidad",
            "im_PrecioVenta",
            "im_ImporteVenta"
        ]
    ].head(10)
)

In [0]:
#Agrupar por producto
product_analysis = (
    df.groupby(
        "cd_NumParte",
        as_index=False
    )
    .agg( #sirve para agrupar por columnas
        cantidad_vendida=(
            "ct_Cantidad",
            "sum"
        ),
        importe_vendido=(
            "im_ImporteVenta",
            "sum"
        )
    )
)

In [0]:
#Top 10 productos
top_productos = (
    product_analysis
    .sort_values(
        "cantidad_vendida",
        ascending=False
    )
    .head(10)
)

display(top_productos)

**Visualización 1**

In [0]:
#Visualización 1
plt.figure(
    figsize=(10, 6)
)

plt.barh(
    top_productos["cd_NumParte"],
    top_productos["cantidad_vendida"]
)

plt.title(
    "Top 10 productos por cantidad vendida"
)

plt.xlabel(
    "Cantidad vendida"
)

plt.ylabel(
    "Número de parte"
)

plt.gca().invert_yaxis()

plt.tight_layout()

plt.show()

#Interpretación

El análisis por número de parte permite identificar cuáles productos presentan mayor movimiento dentro de la muestra disponible.

Los primeros lugares concentran una mayor cantidad de unidades vendidas en comparación con el resto de los productos.

Este resultado permite comenzar a identifciar productos de mayor movimiento, aunque no debe imterpretarse todavía como una clasificación definitiva de rotación debido a que la muestra corresponde a un alcance limitado de datos y temporalidad.

#Inventory Opportunity

**¿qué productos tienen invetnario disponible pero poco o nungún movimiento?**

In [0]:
#Calcular invetnario disponible
df["ct_Disponible"] = (
    df["ct_Existencia"]
    - df["ct_Reservado"]
)

In [0]:
#Agregar inventario por producto
inventory_product = (
    df.groupby(
        "cd_NumParte",
        as_index=False
    )
    .agg(
        cantidad_vendida=(
            "ct_Cantidad",
            "sum"
        ),
        inventario_disponible=(
            "ct_Disponible",
            "first" #toma el inventario correspondiente al producto una sola vez.
        )
    )
)


**oportunidades de revisión de inventario**

In [0]:
#Detectar oportunidades
inventory_opportunities = (
    inventory_product[
        inventory_product[
            "inventario_disponible"
        ] > 0
    ]
    .sort_values(
        [
            "cantidad_vendida",
            "inventario_disponible"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(10)
)

display(inventory_opportunities)

**Visaulización 2**

In [0]:
plt.figure(
    figsize=(10, 6)
)

plt.scatter(
    inventory_product[
        "cantidad_vendida"
    ],
    inventory_product[
        "inventario_disponible"
    ],
    alpha=0.6
)

plt.title(
    "Relación entre ventas e inventario disponible"
)

plt.xlabel(
    "Cantidad vendida"
)

plt.ylabel(
    "Inventario disponible"
)

plt.tight_layout()

plt.show()

**¿Cómo lo interpretamos?**

Piensa en 4 zonas:
- Arriba izquierda, pocas ventas y mucho inventario -> oportunidad de revisión
- Abajo a la derecha, muchas ventas y poco inventario -> posible riesgo de disponibilidad
- Arriba derecha, muchas ventas y mucho inventario -> producto activo con stock alto
- Abajo izquierda, pocas ventas y poco inventario -> producto de poco movimiento y poca existencia

#Customer Analysis

**¿Qué clientes concentran mayor valor de compra?**

In [0]:
#Agrupar por cliente
customer_analysis = (
    df.groupby(
        "cd_IdClienteFC",
        as_index=False
    )
    .agg(
        cantidad_comprada=(
            "ct_Cantidad",
            "sum"
        ),
        importe_comprado=(
            "im_ImporteVenta",
            "sum"
        ),
        productos_distintos=(
            "cd_NumParte",
            "nunique"
        )
    )
)

In [0]:
#Top clientes
top_customers = (
    customer_analysis
    .sort_values(
        "importe_comprado",
        ascending=False
    )
    .head(10)
)

display(top_customers)

Esto responde **¿Qué identificadores de cliente represetnan mayor valor de compra dentro de la muestra?** y conecta directamente con la necesidad de Sell Out de entender quién compra y qué clientes requieren seguimiento.

#Conclusiones

El análisis exploratorio EDA, permitió obtener una primera perspectiva del comportamiento de ventas e inventario de refacciones.

**Productos**

Se identificaron los números de parte con mayor cantidad vendida dentro de la muestra. Estos producots representan los artículos con mayor movimiento observado, aunque los resultados no deben generalizarse todavía al resto de la red de distribuidores.

**Inventario**

La comparación entre cantidad vendida e inventario disponible permitió identifciar productos que pueden requerir revisión debido a que mantienen existencia mientras presetnan un nivel reducido de movimiento.

Estos resultados representan oportunidades de análisis y no una clasificación definitiva de exceso u obsolencia de inventario.

**Clientes**

El análisis mediante cd_IdClienteFC permitió distinguir clientes y ordenar su comportamiento de acuerdo con el importe de compra.

Esta información constituye una base para desarrollar posteriormente segmentaciones de clientes basadas en valor, frecuencia y comportamiento histórico.

**Limitaciones**

El prototipo utiliza una muestra limitada y no contiene suficiente profundidad temporal para realizar análisis de tendencias, abandono de clientes o predicciones confiables de demanda.

Como siguiente etapa sería necesario incorporar múltiples periodos, más distribuidores y reglas de negocio validadas para avanzar hacia indicadores de rotación, cobertura y modelos predictivos.
